# Results Dashboard

Visualizes experiment results from the centralized `results/experiment_results.csv`.

This CSV is automatically populated by `graphium-train` after each run.
Each row = one training run with metadata and all test metrics.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import re
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({'font.size': 12, 'figure.dpi': 100})

RESULTS_CSV = Path("../results/experiment_results.csv")
assert RESULTS_CSV.exists(), f"No results found at {RESULTS_CSV}. Run experiments first."

df = pd.read_csv(RESULTS_CSV)
print(f"Loaded {len(df)} experiment runs")
print(f"Columns: {len(df.columns)}")
df.head()

## 1. Overview: runs by model and task

In [ ]:
print("Runs by model type:")
print(df['model'].value_counts().to_string())
print()
print("Runs by task:")
print(df['task'].value_counts().to_string())
print()
print(f"Finetuned: {df['is_finetuning'].sum()} / Scratch: {(~df['is_finetuning']).sum()}")

## 2. Extract primary metric per task

Each ADMET task has a primary metric. We extract the test metric column for each task.

In [ ]:
# TDC ADMET primary metrics
# https://tdcommons.ai/benchmark/admet_group/overview/
TASK_METRICS = {
    'caco2_wang': ('mae', 'lower'),
    'hia_hou': ('auroc', 'higher'),
    'pgp_broccatelli': ('auroc', 'higher'),
    'bioavailability_ma': ('auroc', 'higher'),
    'lipophilicity_astrazeneca': ('mae', 'lower'),
    'solubility_aqsoldb': ('mae', 'lower'),
    'bbb_martins': ('auroc', 'higher'),
    'ppbr_az': ('mae', 'lower'),
    'vdss_lombardo': ('spearman', 'higher'),
    'cyp2d6_veith': ('auprc', 'higher'),
    'cyp3a4_veith': ('auprc', 'higher'),
    'cyp2c9_veith': ('auprc', 'higher'),
    'cyp2c9_substrate_carbonmangels': ('auprc', 'higher'),
    'cyp2d6_substrate_carbonmangels': ('auprc', 'higher'),
    'cyp3a4_substrate_carbonmangels': ('auroc', 'higher'),
    'half_life_obach': ('spearman', 'higher'),
    'clearance_hepatocyte_az': ('spearman', 'higher'),
    'clearance_microsome_az': ('spearman', 'higher'),
    'ld50_zhu': ('mae', 'lower'),
    'herg': ('auroc', 'higher'),
    'ames': ('auroc', 'higher'),
    'dili': ('auroc', 'higher'),
}

def find_metric_column(df, task, metric_name):
    """Find the test metric column for a given task."""
    candidates = [c for c in df.columns if task in c.lower() and metric_name in c.lower() and 'test' in c.lower()]
    if candidates:
        return candidates[0]
    # Fallback: look for loss/test
    candidates = [c for c in df.columns if 'loss' in c.lower() and 'test' in c.lower()]
    return candidates[0] if candidates else None

def extract_primary_metric(row):
    """Extract the primary metric value for the task in this row."""
    task = row.get('task', '')
    if task not in TASK_METRICS:
        return np.nan
    metric_name, _ = TASK_METRICS[task]
    col = find_metric_column(df, task, metric_name)
    if col is None:
        return np.nan
    return row.get(col, np.nan)

df['primary_metric'] = df.apply(extract_primary_metric, axis=1)
df['metric_direction'] = df['task'].map(lambda t: TASK_METRICS.get(t, (None, None))[1])
df['metric_name'] = df['task'].map(lambda t: TASK_METRICS.get(t, (None, None))[0])

print(f"Tasks with extracted metrics: {df['primary_metric'].notna().sum()} / {len(df)}")

## 3. Dataset type ablation

Compare downstream ADMET performance across pre-training datasets.

In [ ]:
def parse_pretrain_dataset(pretrain_model_path):
    """Infer the pre-training dataset from the checkpoint path."""
    s = str(pretrain_model_path).lower()
    if s == 'scratch' or s == 'nan':
        return 'scratch'
    for name in ['largemix_rxrx3', 'largemix', 'toymix', 'rxrx3']:
        if name in s:
            return name
    return 'unknown'

df['pretrain_type'] = df['pretrain_dataset'].apply(parse_pretrain_dataset)

# Filter to ADMET tasks only
admet_df = df[df['task'].isin(TASK_METRICS.keys())].copy()

if len(admet_df) > 0 and admet_df['primary_metric'].notna().any():
    # Pivot: rows=task, columns=pretrain_type
    pivot = admet_df.pivot_table(
        index='task', columns='pretrain_type', values='primary_metric', aggfunc='mean'
    )
    
    # Reorder columns
    col_order = [c for c in ['scratch', 'toymix', 'largemix', 'rxrx3', 'largemix_rxrx3'] if c in pivot.columns]
    pivot = pivot[col_order]
    
    print("Mean primary metric per task x pre-training dataset:")
    display(pivot.round(4))
else:
    print("No ADMET results found yet. Run experiments first.")

In [ ]:
if len(admet_df) > 0 and admet_df['primary_metric'].notna().any():
    # Normalize metrics for heatmap (higher = better for all)
    pivot_norm = pivot.copy()
    for task in pivot_norm.index:
        direction = TASK_METRICS.get(task, (None, 'higher'))[1]
        if direction == 'lower':
            # Invert so higher = better
            pivot_norm.loc[task] = -pivot_norm.loc[task]
    
    fig, ax = plt.subplots(figsize=(max(8, len(pivot.columns)*2), max(6, len(pivot.index)*0.4)))
    sns.heatmap(
        pivot.round(3), annot=True, fmt='.3f', cmap='RdYlGn',
        linewidths=0.5, ax=ax, center=pivot.median().median()
    )
    ax.set_title('ADMET Performance by Pre-training Dataset Type')
    ax.set_ylabel('ADMET Task')
    ax.set_xlabel('Pre-training Dataset')
    plt.tight_layout()
    plt.show()

## 4. Dataset size ablation

How does the amount of pre-training data affect downstream performance?

In [ ]:
def parse_data_fraction(tags_str):
    """Extract data fraction from W&B tags string."""
    if pd.isna(tags_str):
        return np.nan
    match = re.search(r'frac_(\d+\.?\d*)', str(tags_str))
    if match:
        return float(match.group(1))
    return np.nan

if 'wandb_tags' in df.columns:
    df['data_fraction'] = df['wandb_tags'].apply(parse_data_fraction)
    size_df = df[df['data_fraction'].notna() & df['task'].isin(TASK_METRICS.keys())].copy()
    
    if len(size_df) > 0:
        tasks_with_data = size_df['task'].unique()
        n_tasks = min(len(tasks_with_data), 22)
        ncols = min(6, n_tasks)
        nrows = (n_tasks + ncols - 1) // ncols
        
        fig, axes = plt.subplots(nrows, ncols, figsize=(ncols*3.5, nrows*3), squeeze=False)
        
        for idx, task in enumerate(sorted(tasks_with_data)[:n_tasks]):
            r, c = divmod(idx, ncols)
            ax = axes[r, c]
            
            task_data = size_df[size_df['task'] == task].sort_values('data_fraction')
            metric_name, direction = TASK_METRICS.get(task, ('metric', 'higher'))
            arrow = '\u2191' if direction == 'higher' else '\u2193'
            
            ax.plot(task_data['data_fraction'], task_data['primary_metric'],
                    'o-', color='#466eff', linewidth=2, markersize=6)
            ax.set_title(task, fontsize=10)
            ax.set_xlabel('Fraction')
            if c == 0:
                ax.set_ylabel(f'{metric_name} ({arrow})')
            ax.grid(True, ls='--', alpha=0.4)
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
        
        # Hide unused axes
        for idx in range(n_tasks, nrows * ncols):
            r, c = divmod(idx, ncols)
            axes[r, c].axis('off')
        
        fig.suptitle('Dataset Size Ablation: ADMET Performance vs Pre-training Data Fraction',
                     fontsize=14, y=1.02)
        plt.tight_layout()
        plt.show()
    else:
        print("No size ablation results found. Run 04_ablation_dataset_size.sh first.")
else:
    print("No wandb_tags column found. Size ablation data not available.")

## 5. Aggregate comparison: scratch vs pre-trained

Bar chart comparing mean normalized performance.

In [ ]:
if len(admet_df) > 0 and admet_df['primary_metric'].notna().any():
    # Compute normalized scores (rank within each task, then average)
    def normalize_within_task(group):
        direction = TASK_METRICS.get(group.name, (None, 'higher'))[1]
        vals = group['primary_metric']
        if direction == 'lower':
            vals = -vals
        vmin, vmax = vals.min(), vals.max()
        if vmax == vmin:
            return pd.Series(0.5, index=group.index)
        return (vals - vmin) / (vmax - vmin)
    
    admet_df = admet_df.copy()
    admet_df['normalized_score'] = admet_df.groupby('task', group_keys=False).apply(
        normalize_within_task
    )
    
    agg = admet_df.groupby('pretrain_type')['normalized_score'].agg(['mean', 'std', 'count'])
    order = [c for c in ['scratch', 'toymix', 'largemix', 'rxrx3', 'largemix_rxrx3'] if c in agg.index]
    agg = agg.loc[order]
    
    fig, ax = plt.subplots(figsize=(8, 4))
    colors = ['#999999', '#466eff', '#64B478', '#EB423D', '#FF883D']
    bars = ax.bar(range(len(agg)), agg['mean'], yerr=agg['std'],
                  color=colors[:len(agg)], capsize=5, edgecolor='white', linewidth=1.5)
    ax.set_xticks(range(len(agg)))
    ax.set_xticklabels(agg.index, rotation=15)
    ax.set_ylabel('Normalized Score (higher = better)')
    ax.set_title('Aggregate ADMET Performance by Pre-training Dataset')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(True, axis='y', ls='--', alpha=0.4)
    
    # Add count labels
    for i, (bar, count) in enumerate(zip(bars, agg['count'])):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'n={int(count)}', ha='center', va='bottom', fontsize=9)
    
    plt.tight_layout()
    plt.show()
else:
    print("No ADMET results to aggregate.")

## 6. Per-task detail table

Full results table with all metrics for export.

In [ ]:
if len(admet_df) > 0:
    summary = admet_df[['task', 'model', 'pretrain_type', 'is_finetuning',
                         'unfreeze_depth', 'metric_name', 'primary_metric']].copy()
    summary = summary.sort_values(['task', 'pretrain_type']).reset_index(drop=True)
    display(summary)
    
    # Export
    export_path = Path('../results/summary_table.csv')
    summary.to_csv(export_path, index=False)
    print(f"\nExported summary to {export_path}")

## 7. Raw results preview

Show all available metric columns for debugging.

In [ ]:
metric_cols = [c for c in df.columns if '/test' in c or '/val' in c]
print(f"Available metric columns ({len(metric_cols)}):")
for c in sorted(metric_cols):
    print(f"  {c}")